# Using GEE to create a large set of analysis-ready data in tabular format

Before running this code, run script 01 to create the biotic stacks and then upload the results to GEE (as well as other disturbance stack stuff)

NOTES

~6 hrs to run exports for IDB & SR at 1000m sample with new code

In [39]:
import os

import ee
#import geemap.foliumap as geemap
import geemap

from find_set_root import find_set_project_root
PROJECT_ROOT = find_set_project_root()
print(f"Project root found at: {PROJECT_ROOT}")
import utils.general_functions as ugf


DIR_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DIR_DERIVED = os.path.join(PROJECT_ROOT, 'data', 'derived')
ugf.dir_ensure([DIR_RAW, DIR_DERIVED])

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'
DRIVE_FOLDER = "GEE_Exports_resilience_20251706_v2"

#Prepare to use Earth Engine
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project=GEE_PROJECT)

SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file


#############
# USER SET PARAMETERS
#############

TARGET_CRS = 'EPSG:5070'  # Albers Equal Area - selected since working in W.US
SCALE = 250  # in meters - approximate MODIS resolution
MINIMUM_FOREST_PERC_PIXEL = 0.60  # minimum percent of coarse forest at pixel to be included
MINIMUM_FIRE_PERC_PIXEL = 1  # minimum percent of pixel within fire perimeter to be included as a burnt pixel
MINIMUM_USFS_PERC_PIXEL = 1  # minimum percent of pixel within USFS to be included in data

# Setup aoi
# AOI_REGION = '6.2.14'
# AOI_READABLE = "SRockies"

AOI_REGION = '6.2.9'
AOI_READABLE = "BlueMtns"

# AOI_REGION = '6.2.15'
# AOI_READABLE = "IDBatholith"

# AOI_REGION = '6.2.12'
# AOI_READABLE = "SierraNevada"

# AOI_REGION = '6.2.5'
# AOI_READABLE = "NCascades"

#AOI_READABLE = "test"


# READY
forested_ecoregions_usl3codes = [
    "11",  # Blue Mountains
    "4",  # Cascades
    "1",  # Coast Range
    "9",  # Eastern Cascades Slopes and Foothills
    "78",  # Klamath Mountains
    "77",  # North Cascades
    "2",  # Puget Lowland
    "3",  # Willamette Valley
    "6",  # California Coastal Sage, Chaparral, and    Oak Woodlands
    "5",  # Sierra Nevada
    "8",  # Southern and Baja California Pine-Oak Mountains
    "41",  # Canadian Rockies
    "16",  # Idaho Batholith
    "17",  # Middle Rockies
    "15",  # Columbia Mountains/Northern Rockies
    "21",  # Southern Rockies
    "19",  # Wasatch and Uinta Mountains
    "23",  # Arizona/New Mexico Mountains
    "20",  # Colorado Plateaus
]

forested_ecoregions = [
    {"na_l3name": "Blue Mountains", "na_l3code": "6.2.9", "short_name": "Blue Mtns", "region": "Pacific Northwest", "code_name": "bluemtns", "us_l3code": "11"},
     {"na_l3name": "Cascades", "na_l3code": "6.2.7", "short_name": "Cascades", "region": "Pacific Northwest", "code_name": "cascades", "us_l3code": "4"},
     {"na_l3name": "Coast Range", "na_l3code": "7.1.8", "short_name": "Coast Range", "region": "Pacific Northwest", "code_name": "coastrange", "us_l3code": "1"},
     {"na_l3name": "Eastern Cascades Slopes and Foothills", "na_l3code": "6.2.8", "short_name": "Eastern Cascades", "region": "Pacific Northwest", "code_name": "eastcascades", "us_l3code": "9"},
     {"na_l3name": "Klamath Mountains", "na_l3code": "6.2.11", "short_name": "Klamath Mtns", "region": "Pacific Northwest", "code_name": "klamathmtns", "us_l3code": "78"},
     {"na_l3name": "North Cascades", "na_l3code": "6.2.5", "short_name": "North Cascades", "region": "Pacific Northwest", "code_name": "northcascades", "us_l3code": "77"},
     {"na_l3name": "Straight of Georgia/Puget Lowland", "na_l3code": "7.1.7", "short_name": "Puget Lowland", "region": "Pacific Northwest", "code_name": "pugetlowland", "us_l3code": "2"},
     {"na_l3name": "Willamette Valley", "na_l3code": "7.1.9", "short_name": "Willamette Valley", "region": "Pacific Northwest", "code_name": "willamettevalley", "us_l3code": "3"},
     {"na_l3name": "California Coastal Sage, Chaparral, and Oak Woodlands", "na_l3code": "11.1.1", "short_name": "Central California Mtns", "region": "California", "code_name": "centralcaliforniamtns", "us_l3code": "6"},
     {"na_l3name": "Sierra Nevada", "na_l3code": "6.2.12", "short_name": "Sierra Nevada", "region": "California", "code_name": "sierranevada", "us_l3code": "5"},
     {"na_l3name": "Southern and Baja California Pine-Oak Mountains", "na_l3code": "11.1.3", "short_name": "Southern California Mtns", "region": "California", "code_name": "southerncaliforniamtns", "us_l3code": "8"},
     {"na_l3name": "Canadian Rockies", "na_l3code": "6.2.4", "short_name": "Canadian Rockies", "region": "Upper Rockies", "code_name": "canadianrockies", "us_l3code": "41"},
     {"na_l3name": "Idaho Batholith", "na_l3code": "6.2.15", "short_name": "Idaho Batholith", "region": "Upper Rockies", "code_name": "idahobatholith", "us_l3code": "16"},
     {"na_l3name": "Middle Rockies", "na_l3code": "6.2.10", "short_name": "Middle Rockies", "region": "Upper Rockies", "code_name": "middlerockies", "us_l3code": "17"},
     {"na_l3name": "Columbia Mountains/Northern Rockies", "na_l3code": "6.2.3", "short_name": "Northern Rockies", "region": "Upper Rockies", "code_name": "northernrockies", "us_l3code": "15"},
     {"na_l3name": "Southern Rockies", "na_l3code": "6.2.14", "short_name": "Southern Rockies", "region": "Lower Rockies", "code_name": "southernrockies", "us_l3code": "21"},
     {"na_l3name": "Wasatch and Uinta Mountains", "na_l3code": "6.2.13", "short_name": "Wasatch Uinta Mtns", "region": "Lower Rockies", "code_name": "wasatchuintamtns", "us_l3code": "19"},
     {"na_l3name": "Arizona/New Mexico Mountains", "na_l3code": "13.1.1", "short_name": "AZ/NM Mtns", "region": "Southwest", "code_name": "aznmmtns", "us_l3code": "23"},
     {"na_l3name": "Colorado Plateaus", "na_l3code": "10.1.6", "short_name": "Colorado Plateaus", "region": "Southwest" , "code_name": "coloradoplateaus", "us_l3code": "20"},
]

forested_by_code = {d["us_l3code"]: d for d in forested_ecoregions}



COARSE_FOREST_PERC_MIN = 0
COARSE_TRE_MAX_MIN = 25

FOREST_RANDOM_SAMPLE = 0.2
RANDOM_SEED = 1234
TILE_SIZE = 100000 #100000 works for both 1000 and 500 sample scale
SAMPLE_SCALE = 500  # in meters


Project root found at: C:\Users\tymc5571\dev\compound-disturbance-resilience
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\raw
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived


In [40]:
# Load datasets
chili = ee.Image("CSP/ERGo/1_0/US/CHILI")
lcms = ee.ImageCollection("USFS/GTAC/LCMS/v2024-10")
lcmap = ee.ImageCollection("projects/sat-io/open-datasets/LCMAP/LCPRI")
conservative_forest_annual = ee.Image("projects/ee-tymc5571-cbi-module/assets/conservative_forest_1985_2024")
frg = ee.ImageCollection("LANDFIRE/Fire/FRG/v1_2_0")
tpi = ee.Image("CSP/ERGo/1_0/US/mTPI")
terraclimate = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
topoterra_early = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/TopoTerra_1961-1990")
topoterra_later = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/TopoTerra_1985-2015")
topoterra_normals = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/TopoTerra_1961-2022")
topoterra_future = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/TopoTerra_2C_1985-2015")
srtm = ee.Image("USGS/SRTMGL1_003")
cbi = ee.Image("projects/ee-tymc5571-goodfire/assets/welty_wildfire_cbi_bc_1985_2020")
rap = ee.ImageCollection("projects/rap-data-365417/assets/vegetation-cover-v3")
modis = ee.ImageCollection("MODIS/061/MOD44B")
nfg = ee.ImageCollection("projects/sat-io/open-datasets/USFS/national-forest-group")
biotic = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/biotic_gridded_1km_all_years_severity")
vcf = ee.ImageCollection("MODIS/061/MOD44B")
hd_fingerprint = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_warm_fingerprint")
hd_z_def = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_def")
hd_z_pdsi = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_pdsi")
hd_z_vpd = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_vpd")
hd_z_tmmx = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_tmmx")
hd_z_soil = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_soil")  
hd_z_pr = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_pr")
epa_lvl3 = ee.FeatureCollection("EPA/Ecoregions/2013/L3")
epa_lvl4 = ee.FeatureCollection("EPA/Ecoregions/2013/L4")
fire_ids = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/welty_fire_ids")
usfs = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/usfs_binary")


# AOI
states = ee.FeatureCollection("TIGER/2018/States")

aoi = epa_lvl3.filter(ee.Filter.eq('na_l3code', AOI_REGION))
# aoi= states.filter(ee.Filter.eq('STUSPS', 'CO'))
# aoi = ee.Geometry.Rectangle(
#     coords=[-105.545, 39.948, -105.445, 40.038],  # [xmin, ymin, xmax, ymax]
#     proj=None,
#     geodesic=False
# )
print(aoi.getInfo())

##########################

#Add CBI prefix if not present
cbi = cbi.rename(
    cbi.bandNames().map(
        lambda n: ee.Algorithms.If(
            ee.String(n).slice(0, 4).compareTo('cbi_').eq(0),
            n,
            ee.String('cbi_').cat(n)
        )
    )
)

# The temporal AOI will be based on the years in the CBI data selected
years = cbi.bandNames().map(lambda b: ee.Number.parse(ee.String(b).split('_').get(2)))
print(years.getInfo())
CBI_START_YEAR = ee.Number(years.sort().get(0)).getInfo()
CBI_END_YEAR = ee.Number(years.sort().get(-1)).getInfo()
print(CBI_START_YEAR, CBI_END_YEAR)


{'type': 'FeatureCollection', 'columns': {'l1_key': 'String', 'l2_key': 'String', 'l3_key': 'String', 'na_l1code': 'String', 'na_l1name': 'String', 'na_l2code': 'String', 'na_l2name': 'String', 'na_l3code': 'String', 'na_l3name': 'String', 'shape_area': 'Float', 'shape_leng': 'Float', 'system:index': 'String', 'us_l3code': 'String', 'us_l3name': 'String'}, 'version': 1566852390239791, 'id': 'EPA/Ecoregions/2013/L3', 'properties': {'date_range': [1366070400000, 1366156800000], 'period': 0, 'thumb': 'https://mw1.google.com/ges/dd/images/Ecoregions_thumb.png', 'description': '<p>The U.S. Environmental Protection Agency (USEPA) provides\nthe Ecoregions dataset to serve as a spatial framework for the\nresearch, assessment, management, and monitoring of ecosystems\nand ecosystem components. Ecoregions denote areas of general\nsimilarity in ecosystems and in the type, quality, and quantity\nof environmental resources. These general-purpose regions are\ncritical for structuring and implementin

In [41]:
# Check that the correct ecoregions are selected

# codes_ee = ee.List(forested_ecoregions_usl3codes)
# ecoregions_fc = epa_lvl3.filter(ee.Filter.inList("us_l3code", codes_ee))

# m = geemap.Map()
# m.centerObject(ecoregions_fc, 4)

# style = {
#     "color": "00000000",
#     "width": 2,
#     "fillColor": "FF00FF"
# }
# m.addLayer(ecoregions_fc.style(**style), {}, "Forested EPA L3 ecoregions")
# m

In [42]:
from typing import Literal, Any


# Utility functions

# -------------------------------
# Reclassify an image and mask all values except a specific value (e.g., forest class)
def reclassify_image_binary(value):
    def _reclassify(image):
        binary = image.eq(value).selfMask()
        return binary.copyProperties(image, ['system:time_start'])
    return _reclassify

# -------------------------------
# Combine two collections using logical AND (per image pair), copying time from the first image
def combine_collections_with_and(collection1, collection2, and_name):
    list1 = collection1.toList(collection1.size())
    list2 = collection2.toList(collection2.size())
    combined = list1.zip(list2)

    def combine_pair(pair):
        image1 = ee.Image(ee.List(pair).get(0))
        image2 = ee.Image(ee.List(pair).get(1))
        and_result = image1.And(image2).rename(and_name)
        return and_result.copyProperties(image1, ['system:time_start'])

    return ee.ImageCollection(combined.map(combine_pair))

# -------------------------------
# Add a 'year' property to each image
def add_year_property(image):
    year = image.date().get('year')
    return image.set('year', year)



def resample_to_coarse(source, coarse_image, reducer, max_pixels=4000):
    proj = coarse_image.projection()

    def reduce_and_reproject(img):
        img = img.setDefaultProjection(img.projection())
        return img.reduceResolution(
            reducer=reducer,
            maxPixels=max_pixels
        ).reproject(proj)

    return source.map(reduce_and_reproject) if isinstance(source, ee.ImageCollection) else reduce_and_reproject(source)



def moving_window(
    image: ee.Image,
    window_size: int | float,
    reducer: ee.Reducer,
    kernel_shape: Literal[
        'square', 'circle', 'rect', 'cross', 'gaussian', 'manhattan', 'chebyshev'
    ] = 'square',
    keep_original_bandnames: bool = False,
    **kwargs: Any
) -> ee.Image:
    """
    Apply a moving window operation on an image using a specified reducer, window size, and kernel shape.

    Parameters:
    - image: ee.Image
    - window_size: int or float, diameter in meters
    - reducer: ee.Reducer (e.g., ee.Reducer.mean())
    - kernel_shape: One of ['square', 'circle', 'rect', 'cross', 'gaussian', 'manhattan', 'chebyshev']
    - keep_original_bandnames: If True, retain the original band names after filtering
    - kwargs: Additional parameters:
        - 'xRadius', 'yRadius' for 'rect'
        - 'sigma' for 'gaussian'
        - 'normalize' (bool) for all kernels (default: False)

    Returns:
    - ee.Image: Result of the focal/moving window operation
    """
    normalize: bool = kwargs.get('normalize', False)

    if kernel_shape == 'square':
        kernel = ee.Kernel.square(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'circle':
        kernel = ee.Kernel.circle(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'rect':
        xRadius = kwargs.get('xRadius', window_size / 2)
        yRadius = kwargs.get('yRadius', window_size / 2)
        kernel = ee.Kernel.rect(xRadius=xRadius, yRadius=yRadius, units='meters', normalize=normalize)
    elif kernel_shape == 'cross':
        kernel = ee.Kernel.cross(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'gaussian':
        sigma = kwargs.get('sigma', window_size / 6)
        kernel = ee.Kernel.gaussian(radius=window_size / 2, sigma=sigma, units='meters', normalize=normalize)
    elif kernel_shape == 'manhattan':
        kernel = ee.Kernel.manhattan(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'chebyshev':
        kernel = ee.Kernel.chebyshev(radius=window_size / 2, units='meters', normalize=normalize)
    else:
        raise ValueError(f"Unsupported kernel shape: {kernel_shape}")

    result = image.reduceNeighborhood(reducer=reducer, kernel=kernel)

    if keep_original_bandnames:
        original_names = image.bandNames()
        result = result.rename(original_names)

    return result


def remove_to_bands_append(image):
    """
    Renames bands in the input Earth Engine Image by removing the first part
    (before the first underscore) from each band name.

    Args:
        image (ee.Image): The input image.

    Returns:
        ee.Image: The image with renamed bands.
    """
    old_names = image.bandNames()
    new_names = old_names.map(lambda name: ee.String(name).split('_').slice(1).join('_'))
    return image.rename(new_names)


def remove_to_bands_append2(image, og_ic):
    """
    Removes image ID prefixes (from toBands()) using IDs from the original ImageCollection.
    
    Args:
        image (ee.Image): The image with compound band names after toBands().
        og_ic (ee.ImageCollection): The original ImageCollection before stacking.
    
    Returns:
        ee.Image: Image with cleaned band names.
    """
    ids = ee.List(og_ic.aggregate_array('system:index'))

    def clean_name(name):
        name = ee.String(name)
        def iterate_fn(id_, acc):
            acc_str = ee.String(acc)
            id_str = ee.String(id_).cat('_')
            match = acc_str.slice(0, id_str.length()).equals(id_str)
            return ee.Algorithms.If(match, acc_str.slice(id_str.length()), acc_str)
        return ids.iterate(iterate_fn, name)

    old_names = image.bandNames()
    new_names = old_names.map(lambda b: clean_name(b))
    return image.rename(new_names)

def prepend_or_append_to_band_names(
    image: ee.Image,
    text: str,
    mode: Literal["APPEND", "PREPEND"]
) -> ee.Image:
    """
    Appends or prepends a string to all band names of the given Earth Engine image.

    Args:
        image (ee.Image): The input Earth Engine image.
        text (str): The string to append or prepend.
        mode (Literal["APPEND", "PREPEND"]): Specifies whether to append or prepend the string.

    Returns:
        ee.Image: Image with updated band names.
    """
    old_names = image.bandNames()

    if mode == "APPEND":
        new_names = old_names.map(lambda name: ee.String(name).cat(text))
    elif mode == "PREPEND":
        new_names = old_names.map(lambda name: ee.String(text).cat(name))
    else:
        # This should never be reached due to Literal type, but it's good practice
        raise ValueError("Mode must be either 'APPEND' or 'PREPEND'")

    return image.rename(new_names)


def insert_to_band_names(image: ee.Image, text: str, index: int) -> ee.Image:
    """
    Inserts a string into each band name of an Earth Engine image at the specified index.
    
    Args:
        image (ee.Image): The input image.
        text (str): The string to insert.
        index (int): The character index at which to insert the string.
                     Supports negative indexing from the end.
    
    Returns:
        ee.Image: Image with updated band names.
    """
    def insert_at(name):
        name_str = ee.String(name)
        name_len = name_str.length()
        
        # Compute final insertion index (accounting for negative values)
        insert_pos = ee.Algorithms.If(
            index >= 0,
            ee.Number(index),
            name_len.add(index)  # index is negative, so subtract from length
        )
        insert_pos = ee.Number(insert_pos).clamp(0, name_len)  # avoid out-of-bounds
        
        prefix = name_str.slice(0, insert_pos)
        suffix = name_str.slice(insert_pos)
        return prefix.cat(text).cat(suffix)

    old_names = image.bandNames()
    new_names = old_names.map(insert_at)
    return image.rename(new_names)

def toBands_with_projection(collection):
    collection = ee.ImageCollection(collection)
    image = collection.toBands()
    reference_img = ee.Image(collection.first())
    return image.setDefaultProjection(reference_img.projection())

def filter_bands_by_year(image: ee.Image, first_year: int, last_year: int) -> ee.Image:
    band_names = image.bandNames()

    # Create list of allowed years as strings
    allowed_years = ee.List.sequence(first_year, last_year).map(
        lambda y: ee.Number(y).format('%04d')
    )

    # Map over band names and keep only those that match an allowed year suffix
    def keep_if_valid(band):
        band_str = ee.String(band)
        year_suffix = band_str.slice(-4)
        return ee.Algorithms.If(
            allowed_years.contains(year_suffix),
            band_str,
            None
        )

    # Map and filter out None
    valid_bands = band_names.map(keep_if_valid).removeAll([None])

    return image.select(valid_bands)

In [43]:
#################################
# PREP RESPONSE DATASETS
# Rename bands to include source and year with form vcf_tree_2001 or rap_tree_2015
#################################

####################
# MODIS VCF
####################

def rename_vcf(img):
    year = img.date().format('YYYY')
    new_names = img.bandNames().map(lambda b: ee.String('modis_').cat(b).cat('_').cat(year))
    return img.rename(new_names)

modis_vcf = vcf.select(['Percent_Tree_Cover', 'Percent_Tree_Cover_SD'])
renamed_modis_vcf = modis_vcf.map(rename_vcf)
modis_vcf_im = toBands_with_projection(renamed_modis_vcf)


modis_vcf_im = remove_to_bands_append2(modis_vcf_im, og_ic = renamed_modis_vcf)
modis_og_names = modis_vcf_im.bandNames().getInfo()
modis_new_names = [n.replace("modis_Percent_Tree_Cover", "vcf_tree")
             for n in modis_og_names]
modis_vcf_im = modis_vcf_im.rename(modis_new_names)

print(modis_vcf_im.bandNames().getInfo())

modis_projection = modis_vcf_im.projection()
modis_scale = modis_projection.nominalScale()
print(f"MODIS VCF scale: {modis_scale.getInfo()} meters")
print(modis_projection.getInfo())

vcf_5070 = (modis_vcf_im
            .reproject(
                crs=TARGET_CRS,
                scale=SCALE
            ))

###################
# RAP
###################

def rename_rap(img):
    year = ee.String(img.id().split('/').get(-1))
    new_names = img.bandNames().map(lambda b: ee.String('rap_').cat(b).cat('_').cat(year))
    return img.rename(new_names)

renamed_rap = rap.map(rename_rap)
rap_mosaic = toBands_with_projection(renamed_rap)
fixed_rap_names = rap_mosaic.bandNames().map(lambda b: ee.String(b).slice(5))
rap_mosaic = rap_mosaic.rename(fixed_rap_names)

rap_og_names = rap_mosaic.bandNames().getInfo()
rap_new_names = [n.replace("TRE", "tree") for n in rap_og_names]
rap_mosaic = rap_mosaic.rename(rap_new_names)

# Filter tree bands
tre_bands = rap_mosaic.bandNames().filter(ee.Filter.stringContains('item', 'tree'))
rap_mosaic = rap_mosaic.select(tre_bands)

print(rap_mosaic.bandNames().getInfo())

# Reduce RAP resolution to match MODIS VCF (250m)
rap_5070 = rap_mosaic.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs = TARGET_CRS,
    scale=SCALE
)

['vcf_tree_2000', 'vcf_tree_SD_2000', 'vcf_tree_2001', 'vcf_tree_SD_2001', 'vcf_tree_2002', 'vcf_tree_SD_2002', 'vcf_tree_2003', 'vcf_tree_SD_2003', 'vcf_tree_2004', 'vcf_tree_SD_2004', 'vcf_tree_2005', 'vcf_tree_SD_2005', 'vcf_tree_2006', 'vcf_tree_SD_2006', 'vcf_tree_2007', 'vcf_tree_SD_2007', 'vcf_tree_2008', 'vcf_tree_SD_2008', 'vcf_tree_2009', 'vcf_tree_SD_2009', 'vcf_tree_2010', 'vcf_tree_SD_2010', 'vcf_tree_2011', 'vcf_tree_SD_2011', 'vcf_tree_2012', 'vcf_tree_SD_2012', 'vcf_tree_2013', 'vcf_tree_SD_2013', 'vcf_tree_2014', 'vcf_tree_SD_2014', 'vcf_tree_2015', 'vcf_tree_SD_2015', 'vcf_tree_2016', 'vcf_tree_SD_2016', 'vcf_tree_2017', 'vcf_tree_SD_2017', 'vcf_tree_2018', 'vcf_tree_SD_2018', 'vcf_tree_2019', 'vcf_tree_SD_2019', 'vcf_tree_2020', 'vcf_tree_SD_2020', 'vcf_tree_2021', 'vcf_tree_SD_2021', 'vcf_tree_2022', 'vcf_tree_SD_2022', 'vcf_tree_2023', 'vcf_tree_SD_2023', 'vcf_tree_2024', 'vcf_tree_SD_2024']
MODIS VCF scale: 231.65635826395828 meters
{'type': 'Projection', 'crs': '

In [44]:
##################
# PREP FOREST DATA
##################

###################
# Relaxed forest mask
###################

lcmap_cover = lcmap.filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR))
lcms_cover = lcms.filter(ee.Filter.eq('study_area', 'CONUS')).filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR)).select('Land_Cover')

lcmap_for = lcmap_cover.map(reclassify_image_binary(4))
lcms_for = lcms_cover.map(reclassify_image_binary(1))

combined_forest_mask = lcmap_for.map(lambda img: img.reduce(ee.Reducer.max()).gt(0)).max().Or(
    lcms_for.map(lambda img: img.reduce(ee.Reducer.max()).gt(0)).max()
)
combined_forest = (combined_forest_mask
                        .unmask(0)
                        .rename('forest')
                        .setDefaultProjection(lcmap_cover.first().projection())
)

combined_forest_perc_5070 = combined_forest.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

combined_forest_mask_5070 = (combined_forest_perc_5070
                             .gte(MINIMUM_FOREST_PERC_PIXEL)
                             .selfMask()
                             .unmask(0))


###################
# Create distance from unburned forest the year after (conservative)
###################

cf_names = conservative_forest_annual.bandNames()

# Simply rename bands by subtracting one, e.g. from "forest_1985" to "forest_after_1984"
def rename_to_forest_after(b):
    b = ee.String(b)
    year = ee.Number.parse(b.split('_').get(1))
    new_year = year.subtract(1)
    return ee.String('forest_after_').cat(new_year.format())

new_names = cf_names.map(rename_to_forest_after)

conservative_forest_year_after = conservative_forest_annual.rename(new_names).reproject(
                    crs=TARGET_CRS,
                    scale=30
                )

print(conservative_forest_year_after.bandNames().getInfo())


pixel_size = ee.Number(30)

# Function to get the distance from unburned forest for a given year
# reprojection is done to ensure consistent pixel size of 30 m rather than relying on lazy evaluation
def get_unburned_distance(band_name):
    band_name = ee.String(band_name)
    band = conservative_forest_year_after.select([band_name])

    # Ensure the operation runs at 30 m
    band_30m = band.reproject(crs=TARGET_CRS, scale=pixel_size)

    # Distance^2 in pixels^2 to pixels
    dist_pixels = band_30m.fastDistanceTransform(
        neighborhood=256,
        units='pixels',
        metric='squared_euclidean'
    ).sqrt()

    # Pixels to meters
    dist_meters = dist_pixels.multiply(pixel_size)

    # Freeze the output at 30 m as well
    dist_meters_30m = dist_meters.reproject(crs=TARGET_CRS, scale=pixel_size)

    new_name = ee.String('distance_').cat(band_name)
    return dist_meters_30m.rename(new_name)

distance_ic = ee.ImageCollection(
    conservative_forest_year_after.bandNames().map(get_unburned_distance)
)

unburned_forest_distance = distance_ic.toBands()

# Enforce projection on the final image
unburned_forest_distance = unburned_forest_distance.reproject(
    crs=TARGET_CRS,
    scale=30
)

unburned_forest_distance = remove_to_bands_append(unburned_forest_distance)

print(unburned_forest_distance.bandNames().getInfo())
print(unburned_forest_distance.select(0).projection().getInfo())

unburned_forest_distance_5070 = (unburned_forest_distance
                                .reduceResolution(
                                    reducer=ee.Reducer.mean(),
                                    maxPixels=1024
                                ).reproject(
                                    crs=TARGET_CRS,
                                    scale=SCALE
                                ))

['forest_after_1984', 'forest_after_1985', 'forest_after_1986', 'forest_after_1987', 'forest_after_1988', 'forest_after_1989', 'forest_after_1990', 'forest_after_1991', 'forest_after_1992', 'forest_after_1993', 'forest_after_1994', 'forest_after_1995', 'forest_after_1996', 'forest_after_1997', 'forest_after_1998', 'forest_after_1999', 'forest_after_2000', 'forest_after_2001', 'forest_after_2002', 'forest_after_2003', 'forest_after_2004', 'forest_after_2005', 'forest_after_2006', 'forest_after_2007', 'forest_after_2008', 'forest_after_2009', 'forest_after_2010', 'forest_after_2011', 'forest_after_2012', 'forest_after_2013', 'forest_after_2014', 'forest_after_2015', 'forest_after_2016', 'forest_after_2017', 'forest_after_2018', 'forest_after_2019', 'forest_after_2020', 'forest_after_2021', 'forest_after_2022', 'forest_after_2023']
['distance_forest_after_1984', 'distance_forest_after_1985', 'distance_forest_after_1986', 'distance_forest_after_1987', 'distance_forest_after_1988', 'distanc

In [45]:
#########################
# PREP BIOTIC DATA
#########################

# BIOTIC
# Load the image asset

#print("Available bands:", biotic.bandNames().getInfo())
new_biotic_names = [f'biotic_{year}' for year in range(1997, 2024)]
biotic = biotic.rename(new_biotic_names)

# Create summed biotic layers
biotic_timeseries_sum = biotic.reduce(ee.Reducer.sum()).rename('biotic_timeseries_sum')

biotic_with_sum = biotic.addBands(biotic_timeseries_sum)
biotic_5070 = (biotic_with_sum
.reproject(
    crs=TARGET_CRS,
    scale=1000
))

print("Renamed biotic bands:", biotic_5070.bandNames().getInfo())



###########################
# Normalize by forested area

biotic_years = biotic.bandNames().map(
    lambda b: ee.Number.parse(ee.String(b).split('_').get(-1))
)

BIOTIC_FIRST_YEAR = ee.Number(biotic_years.sort().get(0))
BIOTIC_LAST_YEAR  = ee.Number(biotic_years.sort().get(-1))

def forest_fraction_prior_on_biotic_grid(year: ee.Number, *, max_pixels=4096) -> ee.Image:
    """Return forest fraction (0..1) for (year-1) aggregated onto the biotic_5070 grid."""
    prior = ee.Number(year).subtract(1).format('%d')

    forest_30m = (conservative_forest_annual
        .select(ee.String("forest_").cat(prior))
        .unmask(0)
        .toFloat()
    )

    target_proj = biotic_5070.projection()  # 1km grid (EPSG:5070, scale=1000 per your setup)

    forest_frac = (forest_30m
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=max_pixels)  # mean of 0/1 => fraction
        .reproject(target_proj)
        .rename(ee.String("forestfrac_").cat(prior))
    )
    return forest_frac


def biotic_forestnorm_one_year(year: ee.Number) -> ee.Image:
    """Compute biotic percent-of-forested-area for a single year."""
    y_str = ee.Number(year).format('%d')

    biotic_y = (biotic
        .select(ee.String("biotic_").cat(y_str))
        .toFloat()
    )

    forest_frac_prior = forest_fraction_prior_on_biotic_grid(year)

    # Mask out pixels with no forest in prior year (avoid inf)
    forest_frac_prior = forest_frac_prior.updateMask(forest_frac_prior.gt(0))

    # biotic is already percent of whole pixel (0..100), divide by forest fraction (0..1)
    norm = (biotic_y
        .divide(forest_frac_prior)
        .rename(ee.String("biotic_forestnorm_").cat(y_str))
    )

    # cap at 100
    norm = norm.min(100)

    return norm


years = ee.List.sequence(BIOTIC_FIRST_YEAR, BIOTIC_LAST_YEAR)

biotic_forestnorm_ic = ee.ImageCollection(years.map(lambda y: biotic_forestnorm_one_year(ee.Number(y))))
biotic_forestnorm = biotic_forestnorm_ic.toBands()

biotic_forestnorm = remove_to_bands_append(biotic_forestnorm)

print("Forest-normalized biotic bands:", biotic_forestnorm.bandNames().getInfo())

biotic_5070 = biotic_5070.addBands(biotic_forestnorm)

print("all biotic bands:", biotic_5070.bandNames().getInfo())


Renamed biotic bands: ['biotic_1997', 'biotic_1998', 'biotic_1999', 'biotic_2000', 'biotic_2001', 'biotic_2002', 'biotic_2003', 'biotic_2004', 'biotic_2005', 'biotic_2006', 'biotic_2007', 'biotic_2008', 'biotic_2009', 'biotic_2010', 'biotic_2011', 'biotic_2012', 'biotic_2013', 'biotic_2014', 'biotic_2015', 'biotic_2016', 'biotic_2017', 'biotic_2018', 'biotic_2019', 'biotic_2020', 'biotic_2021', 'biotic_2022', 'biotic_2023', 'biotic_timeseries_sum']
Forest-normalized biotic bands: ['biotic_forestnorm_1997', 'biotic_forestnorm_1998', 'biotic_forestnorm_1999', 'biotic_forestnorm_2000', 'biotic_forestnorm_2001', 'biotic_forestnorm_2002', 'biotic_forestnorm_2003', 'biotic_forestnorm_2004', 'biotic_forestnorm_2005', 'biotic_forestnorm_2006', 'biotic_forestnorm_2007', 'biotic_forestnorm_2008', 'biotic_forestnorm_2009', 'biotic_forestnorm_2010', 'biotic_forestnorm_2011', 'biotic_forestnorm_2012', 'biotic_forestnorm_2013', 'biotic_forestnorm_2014', 'biotic_forestnorm_2015', 'biotic_forestnorm_2

In [46]:
# Test BIOTIC data
test_area2 = (
    ee.FeatureCollection("TIGER/2018/Counties")
    .filter(ee.Filter.eq("NAME", "Jefferson"))
    .filter(ee.Filter.eq("STATEFP", "08"))  # Colorado
    .geometry()
)

biotic_test = biotic_5070.clip(test_area2).select("biotic_2005")
biotic_forestnorm_test = biotic_forestnorm.clip(test_area2).select("biotic_forestnorm_2005")
conservative_forest_annual_test = conservative_forest_annual.clip(test_area2).select("forest_2004")

m2 = geemap.Map(center=[40.05, -105.35], zoom=9)
m2.addLayer(biotic_test, {"min": 0, "max": 10}, "Biotic 2005")
m2.addLayer(biotic_forestnorm_test, {"min": 0, "max": 10}, "Biotic Forest-Normalized 2005")
m2.addLayer(conservative_forest_annual_test, {"min": 0, "max": 1}, "Conservative Forest 2004")
m2

Map(center=[40.05, -105.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [47]:
nfg_5070 = (nfg
            .filter(ee.Filter.eq('id_no', 'conus_forestgroup'))
            .first()
            .select("b1")
            .rename('nfg')
            .reproject(
                crs=TARGET_CRS,
                scale=SCALE
            )
)

## NOTES TO SELF : could backfill NFG values where missing with a 1km modal kernel, 
# but won't do that now; check how many points we have to drop due to mising NFG

In [48]:
#################################
# PREP MODERATOR DATSETS
#################################

###################
# CHILI, TPI, SRTM
###################

srtm_5070 = srtm.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

tpi_5070 = tpi.resample(
    mode='bilinear'
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

chili_5070 = chili.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

###################
# FRG
###################

# Reclassify FRG to simplify non-regime classes
frg_img = frg.filterMetadata('system:index', 'contains', 'CONUS').first()
frg_reclass = frg_img.remap(
    [1, 2, 3, 4, 5, 111, 112, 131, 132, 133],
    [1, 2, 3, 4, 5, 6, 6, 6, 6, 6],
    0, 'FRG'
).rename('frg_reclass')

frg_reclass_5070 = frg_reclass.reduceResolution(
    reducer=ee.Reducer.mode(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

###################
# HD
###################
hd_fingerprint_5070 = filter_bands_by_year(hd_fingerprint, CBI_START_YEAR, CBI_END_YEAR).reproject(
    crs=TARGET_CRS,
    scale=4000
)

###################
# Topoterra
###################

# AET (b1) and DEF (b2) for each time slice
tt_early_aet   = topoterra_early.select('b1').rename('tt_early_aet')
tt_early_def   = topoterra_early.select('b2').rename('tt_early_def')

tt_later_aet   = topoterra_later.select('b1').rename('tt_later_aet')
tt_later_def   = topoterra_later.select('b2').rename('tt_later_def')

tt_normals_aet = topoterra_normals.select('b1').rename('tt_normal_aet')
tt_normals_def = topoterra_normals.select('b2').rename('tt_normal_def')

tt_future_aet  = topoterra_future.select('b1').rename('tt_future_aet')
tt_future_def  = topoterra_future.select('b2').rename('tt_future_def')

# Change bands: early - future
tt_change_aet = tt_future_aet.subtract(tt_early_aet).rename('tt_change_aet')
tt_change_def = tt_future_def.subtract(tt_early_def).rename('tt_change_def')

# Combine everything into one image
topoterra_image = ee.Image.cat([
    tt_early_aet, tt_early_def,
    tt_later_aet, tt_later_def,
    tt_normals_aet, tt_normals_def,
    tt_future_aet, tt_future_def,
    tt_change_aet, tt_change_def
])

print("Topoterra bands:", topoterra_image.bandNames().getInfo())

topoterra_5070 = topoterra_image.reproject(
    crs=TARGET_CRS,
    scale=SCALE)

###################
# TERRACLIMATE
###################

# Terraclimate AET and DEF
terraclimate_mean_5070 = terraclimate.select(['aet', 'def']) \
    .filter(ee.Filter.calendarRange(1984, 2020, 'year')) \
    .mean() \
    .reproject(crs=TARGET_CRS,
               scale=4000)

# Define the functions to compute annual and summer terraclimate means
def annual_terraclimate_image(index):

    def annual_terraclimate_images(year):
        year = ee.Number(year).toInt()
        mean_image = terraclimate \
            .filter(ee.Filter.calendarRange(year, year, 'year')) \
            .select(index) \
            .mean() \
            .rename(ee.String(index).cat('_annual_').cat(year.format('%d')))
        return mean_image

    annual_images = years.map(annual_terraclimate_images)
    annual_collection = ee.ImageCollection(annual_images)
    annual_image = toBands_with_projection(annual_collection)
    annual_image = remove_to_bands_append(annual_image)
    return(annual_image)

def summer_terraclimate_image(index):
    def summer_mean_image(year):
        year = ee.Number(year)
        # Filter for June (6), July (7), August (8) of that year
        summer_months = terraclimate \
            .filter(ee.Filter.calendarRange(year, year, 'year')) \
            .filter(ee.Filter.calendarRange(6, 8, 'month')) \
            .select(index) \
            .mean() \
            .rename(ee.String(index).cat('_summer_').cat(year.format()))
        return summer_months

    summer_images = years.map(summer_mean_image)
    summer_collection = ee.ImageCollection(summer_images)
    summer_image = toBands_with_projection(summer_collection)
    summer_image = remove_to_bands_append(summer_image)
    return summer_image


pdsi_annual_5070 = (annual_terraclimate_image('pdsi').
               reproject(
                    crs=TARGET_CRS,
                    scale=4000
                ))
# pdsi_summer_5070 = (summer_terraclimate_image('pdsi').
#                reproject(
#                     crs=TARGET_CRS,
#                     scale=4000
#                 ))
# vpd_annual_5070 = (annual_terraclimate_image('vpd').
#                reproject(
#                     crs=TARGET_CRS,
#                     scale=4000
#                 ))
# vpd_summer_5070 = (summer_terraclimate_image('vpd').
#                reproject(
#                     crs=TARGET_CRS,
#                     scale=4000
#                 ))

print("PDSI annual bands:", pdsi_annual_5070.bandNames().getInfo())

Topoterra bands: ['tt_early_aet', 'tt_early_def', 'tt_later_aet', 'tt_later_def', 'tt_normal_aet', 'tt_normal_def', 'tt_future_aet', 'tt_future_def', 'tt_change_aet', 'tt_change_def']
PDSI annual bands: ['pdsi_annual_1997', 'pdsi_annual_1998', 'pdsi_annual_1999', 'pdsi_annual_2000', 'pdsi_annual_2001', 'pdsi_annual_2002', 'pdsi_annual_2003', 'pdsi_annual_2004', 'pdsi_annual_2005', 'pdsi_annual_2006', 'pdsi_annual_2007', 'pdsi_annual_2008', 'pdsi_annual_2009', 'pdsi_annual_2010', 'pdsi_annual_2011', 'pdsi_annual_2012', 'pdsi_annual_2013', 'pdsi_annual_2014', 'pdsi_annual_2015', 'pdsi_annual_2016', 'pdsi_annual_2017', 'pdsi_annual_2018', 'pdsi_annual_2019', 'pdsi_annual_2020', 'pdsi_annual_2021', 'pdsi_annual_2022', 'pdsi_annual_2023']


In [49]:
###################
# PREP USFS DATA
###################

usfs_perc_5070 = usfs.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

usfs_mask_5070 = (usfs_perc_5070
                  .gte(MINIMUM_USFS_PERC_PIXEL)
                  .selfMask())

In [50]:
###################
# PREP FIRE DATA
###################

print("CBI bands:", cbi.bandNames().getInfo())

# Fire mask
fire_mask = (cbi
             .mask()
             .reduce(ee.Reducer.sum().unweighted())
             .gt(0)
             .rename('valid')
             .unmask(0))

fire_mask_perc_5070 = fire_mask.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

fire_mask_5070 = (fire_mask_perc_5070
                  .gte(MINIMUM_FIRE_PERC_PIXEL)
                  .selfMask()
                  .unmask(0))



# CBI filled versions
cbi_filled_neg = cbi.unmask(-1)
cbi_filled_zeros = cbi.unmask(0)

# CBI resampled to 5070 at SCALE
cbi_filled_zeros_5070 = cbi_filled_zeros.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

cbi_filled_neg_5070 = cbi_filled_neg.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

########################
# CREATE MOVING WINDOW & RESAMPLED DATASETS
########################

# CBI Moving window
cbi_mw_mean = moving_window(image=cbi_filled_zeros, window_size=500, kernel_shape='circle', reducer=ee.Reducer.mean(), keep_original_bandnames=True)
cbi_mw_mean = insert_to_band_names(image = cbi_mw_mean, text='500mmwmean_', index=-9)
cbi_mw_mean_5070 = cbi_mw_mean.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

cbi_mw_std = moving_window(image=cbi_filled_zeros, window_size=500, kernel_shape='circle', reducer=ee.Reducer.stdDev(), keep_original_bandnames=True)
cbi_mw_std = insert_to_band_names(image=cbi_mw_std, text='500mmwstd_', index=-9)
cbi_mw_std_5070 = cbi_mw_std.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
).reproject(
    crs=TARGET_CRS,
    scale=SCALE
)

CBI bands: ['cbi_year_1985', 'cbi_year_1986', 'cbi_year_1987', 'cbi_year_1988', 'cbi_year_1989', 'cbi_year_1990', 'cbi_year_1991', 'cbi_year_1992', 'cbi_year_1993', 'cbi_year_1994', 'cbi_year_1995', 'cbi_year_1996', 'cbi_year_1997', 'cbi_year_1998', 'cbi_year_1999', 'cbi_year_2000', 'cbi_year_2001', 'cbi_year_2002', 'cbi_year_2003', 'cbi_year_2004', 'cbi_year_2005', 'cbi_year_2006', 'cbi_year_2007', 'cbi_year_2008', 'cbi_year_2009', 'cbi_year_2010', 'cbi_year_2011', 'cbi_year_2012', 'cbi_year_2013', 'cbi_year_2014', 'cbi_year_2015', 'cbi_year_2016', 'cbi_year_2017', 'cbi_year_2018', 'cbi_year_2019', 'cbi_year_2020']


In [51]:
# # # TEST FIXED DATA GENERATION STEPS - CBI manipulations

# # Approx. town center of Bailey, CO
# bailey_pt = ee.Geometry.Point([-105.4733328, 39.4055449])

# # 25 km radius buffer (i.e., 25 km in every direction)
# test_area = bailey_pt.buffer(25_000)

# # # Use visualization copies here
# # modis_clipped = vcf_5070.clip(test_area).select("vcf_tree_2000")
# # rap_mosaic_clipped = rap_mosaic.clip(test_area).select("rap_tree_2000")
# # rap_reduced_clipped = rap_5070.clip(test_area).select("rap_tree_2000")
# # biotic_clipped = biotic_5070.clip(test_area).select("biotic_2000")
# # combined_forest_clipped = combined_forest.clip(test_area)
# # combined_forest_perc_5070_clipped = combined_forest_perc_5070.clip(test_area)
# # combined_forest_mask_5070_clipped = combined_forest_mask_5070.clip(test_area)
# # nfg_clipped = nfg_5070.clip(test_area)
# # conservative_forest_annual_clipped = conservative_forest_year_after.clip(test_area).select("forest_after_2000")
# # unburned_forest_distance_clipped = unburned_forest_distance.clip(test_area).select("distance_forest_after_2000")
# # unburned_forest_distance_5070_clipped = unburned_forest_distance_5070.clip(test_area).select("distance_forest_after_2000")

# cbi_clipped = cbi.clip(test_area).select("cbi_year_2000")
# cbi_5070_clipped = cbi_filled_zeros_5070.clip(test_area).select("cbi_year_2000")
# cbi_mw_mean_clipped = cbi_mw_mean.clip(test_area).select("cbi_500mmwmean_year_2000")
# cbi_mw_std_clipped = cbi_mw_std.clip(test_area).select("cbi_500mmwstd_year_2000")
# cbi_mw_mean_5070_clipped = cbi_mw_mean_5070.clip(test_area).select("cbi_500mmwmean_year_2000")
# cbi_mw_std_5070_clipped = cbi_mw_std_5070.clip(test_area).select("cbi_500mmwstd_year_2000")

# # fire_mask_5070_clipped = fire_mask_5070.clip(test_area).select("valid")
# # fire_mask_perc_5070_clipped = fire_mask_perc_5070.clip(test_area).select("valid")

# cbi_vis = {
#     "min": 0,
#     "max": 3,
#     "palette": [
#         "#0000ff",  # 0
#         "#00ff00",
#         "#ffff00",
#         "#ff7f00",
#         "#ff0000"   # 3
#     ]
# }

# m = geemap.Map(center=[40.05, -105.35], zoom=9)
# # # m.addLayer(modis_clipped, {"min": 0, "max": 100}, "MODIS vcf_tree")
# # # m.addLayer(rap_mosaic_clipped, {"min": 0, "max": 100}, "RAP mosaic rap_tree")
# # # m.addLayer(rap_reduced_clipped, {"min": 0, "max": 100}, "RAP reduced rap_tree")
# # # m.addLayer(biotic_clipped, {"min": 0, "max": 1}, "Biotic 2010")
# # # m.addLayer(combined_forest_clipped, {"min": 0, "max": 1}, "Combined forest")
# #m.addLayer(combined_forest_mask_5070_clipped, {"min": 0, "max": 1}, "Combined forest mask 5070")
# # # m.addLayer(nfg_clipped, {"min": 0, "max": 1}, "NFG 5070")
# # # m.addLayer(unburned_forest_distance_clipped, {"min": 0, "max": 1000}, "Unburned forest distance")
# # # m.addLayer(conservative_forest_annual_clipped, {"min": 0, "max": 1}, "Conservative forest annual 2000")
# # # m.addLayer(unburned_forest_distance_5070_clipped, {"min": 0, "max": 1000}, "Unburned forest distance 5070")
# m.addLayer(cbi_clipped, cbi_vis, "CBI 2000")   
# m.addLayer(cbi_5070_clipped, cbi_vis, "CBI 5070 2000")
# m.addLayer(cbi_mw_mean_clipped, cbi_vis, "CBI 500mmwmean 2000")
# m.addLayer(cbi_mw_std_clipped, cbi_vis, "CBI 500mmwstd 2000")
# m.addLayer(cbi_mw_mean_5070_clipped, cbi_vis, "CBI 500mmwmean 5070 2000")
# m.addLayer(cbi_mw_std_5070_clipped, cbi_vis, "CBI 500mmwstd 5070 2000")
# # m.addLayer(fire_mask_perc_5070_clipped, {"min": 0, "max": 1}, "Fire mask perc 5070")
# # m.addLayer(fire_mask_5070_clipped, {"min": 0, "max": 1}, "Fire mask 5070")
# m
# # m.addLayerControl()                                                                        #

In [52]:
# CREATE SAMPLE

# # Mask to the ROI
# coords_img = ee.Image.pixelCoordinates(biotic_sum.projection())
# masked_coords = coords_img.updateMask(biotic_sum.mask())

# # Sample pixel centers (creates FeatureCollection with geometries)
# sample_points = masked_coords.sample(
#     region=aoi,
#     scale=1000,
#     geometries=True
# )


# print('test sample size:', sample_points.size().getInfo())

# Map = geemap.Map(center=[0, 0], zoom=2)
# Map.addLayer(biotic_sum_1997_2007, {'min': 0, 'max': 100, 'palette': ['blue', 'yellow', 'red']}, 'Biotic Sum 1997-2007')
# Map.addLayer(sample_points, {}, 'Sample Points')
# Map

In [53]:
# Map = geemap.Map(center=[40.0, -105.5], zoom=11)

# # Add forest percentage image
# Map.addLayer(forest_perc_coarse, {
#     'min': 0, 'max': 100, 'palette': ['white', 'green']
# }, 'Forest % (Coarse)')

# Map.addLayer(fire_mask, {
#     'min': 0, 'max': 1, 'palette': ['white', 'red']
# }, 'Fire Mask')

# # Add forest mask
# Map.addLayer(combined_forest_mask.updateMask(combined_forest_mask), {
#     'palette': ['darkgreen']
# }, 'Combined Forest Mask')

# # Add coarse sample points
# #Map.addLayer(fc_coarse, {}, 'Coarse Sample Points')

#Map


In [54]:
# # EXTRACTION

# fc = sample_points
# fc_coarse = sample_points

# # Do direct extraction and filter of forest and fire
# # fc = extract_single_band_value(combined_forest_mask, fc, 'forest').filter(ee.Filter.eq('forest', 1))
# # fc = extract_single_band_value(fire_mask, fc, 'fire')

# fire_only = fc.filter(ee.Filter.And(
#     ee.Filter.eq('forest', 1),
#     ee.Filter.eq('fire', 1)
# ))
# forest_only = fc.filter(ee.Filter.And(
#     ee.Filter.eq('forest', 1),
#     ee.Filter.eq('fire', 0)
# ))

# forest_sample = forest_only.randomColumn('rand', seed=RANDOM_SEED).filter(ee.Filter.lt('rand', FOREST_RANDOM_SAMPLE))
# combined = fire_only.merge(forest_sample)

# # Add variables to points
# # Define (image, label) pairs
# image_label_pairs = [
#     (srtm, 'srtm'),
#     (tpi, 'tpi'),
#     (frg_reclass, 'frg_reclass'),
#     (chili, 'chili'),
#     (lf_bps_img, 'lf_bps'),
#     (biotic_sum, 'biotic_sum')
# ]

# # Apply the extraction function
# for img, label in image_label_pairs:
#     fc = extract_single_band_value(img, fc, label)



# # Add multi-band data using reduceRegions
# # List of images to reduce
# multiband_images = [cbi, biotic, rap_mosaic, treemap, terraclimate_mean, \
#                     pdsi_annual, pdsi_summer, \
#                     # vpd_annual, vpd_summer, \
#                     hd_fingerprint, \
#                     # hd_z_def, hd_z_pdsi, hd_z_vpd, hd_z_tmmx, hd_z_soil, hd_z_pr, \
#                     cbi_mw_median, cbi_mw_std, unburned_forest_distance]

# # Apply reduceRegions for each image
# for img in multiband_images:
#     fc = img.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=30)


# # Do coarse extraction
# #fc_coarse = extract_single_band_value(forest_perc_coarse, fc_coarse, 'forest').filter(ee.Filter.gt('forest', COARSE_FOREST_PERC_MIN))
# #fc_coarse = extract_single_band_value(forest_cover_max, fc_coarse, 'forest').filter(ee.Filter.gt('forest', COARSE_TRE_MAX_MIN))


# # Add variables to coarse points
# # Define (image, label) pairs for coarse extraction
# image_label_pairs_coarse = [
#     #(fire_perc_coarse, 'fire'),
#     (srtm_coarse, 'srtm'),
#     (tpi_coarse, 'tpi'),
#     (frg_reclass_coarse, 'frg_reclass'),
#     (chili_coarse, 'chili'),
#     (lf_bps_img_coarse, 'lf_bps'),
#     (biotic_sum, 'biotic_sum')
# ]


# # Apply the extraction function
# for img, label in image_label_pairs_coarse:
#     fc_coarse = extract_single_band_value(img, fc_coarse, label)


# # Multiband extraction for coarse data
# multiband_images_coarse = [ cbi_coarse, biotic, rap_mosaic_coarse, treemap_coarse, terraclimate_mean, \
#     pdsi_annual, pdsi_summer, \
#     # vpd_annual, vpd_summer, \
#     hd_fingerprint, \
#     # hd_z_def, hd_z_pdsi, hd_z_vpd, hd_z_tmmx, hd_z_soil, hd_z_pr, \
#     cbi_mw_median_coarse, cbi_mw_std_coarse, unburned_forest_distance_coarse]

# # Apply reduceRegions for each image
# for img in multiband_images_coarse:
#     fc_coarse = img.reduceRegions(collection=fc_coarse, reducer=ee.Reducer.first(), scale=30)





# # Round to minimize string sizes in CSVs
# def round_fc(feature_collection):
#     def round_feature(feat):
#         props = feat.toDictionary()
#         keys = ee.List(props.keys())

#         def keep_if_tenths(key):
#             key_str = ee.String(key)
#             cond1 = key_str.slice(0, 7).equals('biotic_')
#             cond2 = key_str.slice(0, 4).equals('cbi_')
#             return ee.Algorithms.If(cond1, key, ee.Algorithms.If(cond2, key, None))

#         def keep_if_ints(key):
#             key_str = ee.String(key)
#             cond1 = key_str.slice(0, 4).equals('aet')
#             cond2 = key_str.slice(0, 4).equals('def')
#             cond3 = key_str.slice(0, 9).equals('distance_')
#             cond4 = key_str.slice(0, 5).equals('pdsi_')
#             cond5 = key_str.slice(0, 4).equals('vpd_')
#             return ee.Algorithms.If(cond1, key,
#                 ee.Algorithms.If(cond2, key,
#                 ee.Algorithms.If(cond3, key,
#                 ee.Algorithms.If(cond4, key,
#                 ee.Algorithms.If(cond5, key, None)))))

#         round_tenths = keys.map(keep_if_tenths).removeAll([None])
#         round_int = keys.map(keep_if_ints).removeAll([None])

#         def round_tenths_fn(key):
#             val = ee.Number(props.get(key)).multiply(10).round().divide(10)
#             return ee.List([key, val])

#         def round_int_fn(key):
#             val = ee.Number(props.get(key)).round()
#             return ee.List([key, val])

#         updates_tenths = round_tenths.map(round_tenths_fn)
#         updates_int = round_int.map(round_int_fn)

#         updates_tenths_dict = ee.Dictionary(ee.List(updates_tenths).flatten())
#         updates_int_dict = ee.Dictionary(ee.List(updates_int).flatten())

#         all_updates = updates_tenths_dict.combine(updates_int_dict, overwrite=True)
#         return feat.set(all_updates)

#     return feature_collection.map(round_feature)

# fc = round_fc(fc)
# fc_coarse = round_fc(fc_coarse)


In [55]:
# Test the feature collections

# first_feature = fc.first()
# property_names = first_feature.propertyNames()
# print('Property names:', property_names.getInfo())
# print(first_feature.getInfo())
# print(len(property_names.getInfo()))

# first_coarse_feature = fc_coarse.first()
# coarse_property_names = first_coarse_feature.propertyNames()
# print('Coarse property names:', coarse_property_names.getInfo())
# print(first_coarse_feature.getInfo())
# print(len(coarse_property_names.getInfo()))

In [56]:


##################### DEPRECATED - DOESN'T BATCH GEOGRAPHICALLY #####################

# # Get estimated size of the feature collection based on area of forest in AOI

# # Mask the image to retain only forest (value == 1)
# binary = combined_forest_mask.updateMask(combined_forest_mask.eq(1)).rename('forest')

# # Multiply by pixel area to get area per pixel
# area_image = binary.multiply(ee.Image.pixelArea())

# # Reduce to total area in square meters
# area_stats = area_image.reduceRegion(
#     reducer=ee.Reducer.sum(),
#     geometry=aoi,
#     scale=30,
#     maxPixels=1e13
# )

# # Convert to square kilometers
# forest_area_km2 = ee.Number(area_stats.get('forest')).divide(1e6)
# estimated_partitions = forest_area_km2.multiply(FOREST_RANDOM_SAMPLE).divide(BATCH_SIZE).ceil()

# # Add reproducible random column
# fc = fc.randomColumn('batch_rand', seed=RANDOM_SEED)

# # Define number of partitions (based on estimated size or safe guess)
# num_partitions = estimated_partitions.getInfo()
# step = 1.0 / num_partitions
# print(f"Estimated partitions: {num_partitions}, Step size: {step}")

# # Export loop
# for i in range(num_partitions):
#     print(f"Starting export for batch {i + 1} of {num_partitions}")
#     lower = i * step
#     upper = (i + 1) * step
#     subset = fc.filter(ee.Filter.And(
#         ee.Filter.gte('batch_rand', lower),
#         ee.Filter.lt('batch_rand', upper)
#     ))

#     task = ee.batch.Export.table.toDrive(
#         collection=subset,
#         description=f'fc_export_batch_{AOI_READABLE}_btchsz{BATCH_SIZE}_{i}',
#         folder=DRIVE_FOLDER,
#         fileNamePrefix=f'fc_batch_{AOI_READABLE}_btchsz{BATCH_SIZE}_{i}',
#         fileFormat='CSV'
#     )
#     task.start()






########## DEPRECATED - NEED TO BATCH ############
# # Export fine-resolution feature collection
# task_fc = ee.batch.Export.table.toDrive(
#     collection=fc,
#     description=f'fc_{AOI_READABLE}',
#     fileFormat='CSV',
#     folder=DRIVE_FOLDER, 
#     fileNamePrefix=f'fc_{AOI_READABLE}'
# )
# task_fc.start()

# # Export coarse-resolution feature collection
# task_fc_coarse = ee.batch.Export.table.toDrive(
#     collection=fc_coarse,
#     description= f'fc_coarse_{AOI_READABLE}',
#     fileFormat='CSV',
#     folder=DRIVE_FOLDER,
#     fileNamePrefix= f'fc_coarse_{AOI_READABLE}'
# )
# task_fc_coarse.start()


In [57]:

# Used in extract_additional_data to get lat/long from point features
def add_latlon(feature):
    coords = feature.geometry().transform('EPSG:4326', 1).coordinates() #transform to WGS84 to extract lat/long
    return feature.set({
        'long': coords.get(0),
        'lat': coords.get(1)
    })

# Used in extract_additional_data to extract single band values
def extract_single_band_value(image, collection, new_name):
    new_pts = image.reduceRegions(
        collection=collection,
        reducer=ee.Reducer.first(),
        scale=30
    ).map(lambda pt: pt.set(new_name, pt.get('first')).set('first', None))
    return new_pts


# used in extract_additional_data to keep csv size smaller by rounding certain variables
# note that ugly logic is used because we are using ee objects
def round_fc(feature_collection):
    def round_feature(feat):
        props = feat.toDictionary()
        keys = ee.List(props.keys())

        def keep_if_tenths(key):
            key_str = ee.String(key)
            cond1 = key_str.slice(0, 7).equals('biotic_')
            cond2 = key_str.slice(0, 4).equals('cbi_')
            return ee.Algorithms.If(cond1, key, ee.Algorithms.If(cond2, key, None))

        def keep_if_ints(key):
            key_str = ee.String(key)
            cond1 = key_str.slice(0, 4).equals('aet')
            cond2 = key_str.slice(0, 4).equals('def')
            cond3 = key_str.slice(0, 9).equals('distance_')
            cond4 = key_str.slice(0, 5).equals('pdsi_')
            cond5 = key_str.slice(0, 4).equals('vpd_')
            cond6 = key_str.slice(0, 3).equals('tt_')
            cond7 = key_str.slice(0, 4).equals('srtm')
            cond8 = key_str.slice(0, 3).equals('tpi')
            cond9 = key_str.slice(0, 5).equals('chili')
            cond10 = key_str.slice(0, 9).equals('rap_tree_')
            return ee.Algorithms.If(cond1, key,
                ee.Algorithms.If(cond2, key,
                ee.Algorithms.If(cond3, key,
                ee.Algorithms.If(cond4, key,
                ee.Algorithms.If(cond5, key,
                ee.Algorithms.If(cond6, key,
                ee.Algorithms.If(cond7, key,
                ee.Algorithms.If(cond8, key,
                ee.Algorithms.If(cond9, key,
                ee.Algorithms.If(cond10, key, None))))))))))

        round_tenths = keys.map(keep_if_tenths).removeAll([None])
        round_int = keys.map(keep_if_ints).removeAll([None])

        def round_tenths_fn(key):
            val = ee.Number(props.get(key)).multiply(10).round().divide(10)
            return ee.List([key, val])

        def round_int_fn(key):
            val = ee.Number(props.get(key)).round()
            return ee.List([key, val])

        updates_tenths = round_tenths.map(round_tenths_fn)
        updates_int = round_int.map(round_int_fn)

        updates_tenths_dict = ee.Dictionary(ee.List(updates_tenths).flatten())
        updates_int_dict = ee.Dictionary(ee.List(updates_int).flatten())

        all_updates = updates_tenths_dict.combine(updates_int_dict, overwrite=True)
        return feat.set(all_updates)

    return feature_collection.map(round_feature)



# Function to extract additional data for points of interest from both single-band and multi-band images
def extract_additional_data(fc_interest, tile_geom):
    # Add variables to points
    # Define (image, label) pairs
    image_label_pairs = [
        (srtm_5070, 'srtm'),
        (tpi_5070, 'tpi'),
        (frg_reclass_5070, 'frg_reclass'),
        (nfg_5070, 'nfg'),
        (chili_5070, 'chili')
    ]

    # Apply the extraction function
    for img, label in image_label_pairs:
        img_clipped = img.clip(tile_geom)
        fc_interest = extract_single_band_value(img_clipped, fc_interest, label)



    # Add multi-band data using reduceRegions
    # List of images to reduce
    multiband_images = ([biotic_5070,
                        terraclimate_mean_5070, topoterra_5070,
                        cbi_filled_neg_5070, # use filled version to avoid NaNs
                        cbi_mw_mean_5070, cbi_mw_std_5070, 
                        rap_5070, vcf_5070, #response variables
                        pdsi_annual_5070, #pdsi_summer_5070,
                        # vpd_annual, vpd_summer,
                        hd_fingerprint_5070,
                        # hd_z_def, hd_z_pdsi, hd_z_vpd, hd_z_tmmx, hd_z_soil, hd_z_pr,
                        unburned_forest_distance_5070])

    # Apply reduceRegions for each image
    for img_m in multiband_images:
        img_clipped_m = img_m.clip(tile_geom)
        fc_interest = img_clipped_m.reduceRegions(collection=fc_interest, reducer=ee.Reducer.first(), scale=30, tileScale = 4)

    fc_interest = round_fc(fc_interest)
    fc_interest = fc_interest.map(add_latlon)

    # fc_interest = attach_polygon_properties_to_points(
    #     samples_fc=fc_interest,
    #     polygons_fc=epa_lvl4,
    #     property_names=['us_l3code', 'us_l3name', 'us_l4code', 'us_l4name'])

    return fc_interest


#######################################################
# PRIMARY OPERATING FUNCTION FOR TILE-BASED EXTRACTION
#######################################################
def extract(tile_geom):
    target_tile = vcf_5070.clip(tile_geom).select(0).add(1)  # add 1 to avoid zero-mask issues; use the first vcf band
    coords_img = ee.Image.pixelCoordinates(target_tile.projection())
    masked_coords = coords_img.updateMask(target_tile.mask())

    # Sample pixel centers (creates FeatureCollection with geometries)
    sample_points = masked_coords.sample(
        region=tile_geom,
        scale=SAMPLE_SCALE,
        geometries=True
    )

    fc = sample_points

    # Limit sample to forested USFS lands. Take a subsample of unburned areas and all burned areas.
    fc = extract_single_band_value(usfs_mask_5070, fc, 'usfs').filter(ee.Filter.eq('usfs', 1))
    fc = extract_single_band_value(combined_forest_mask_5070, fc, 'forest').filter(ee.Filter.eq('forest', 1))
    fc = extract_single_band_value(fire_mask_5070, fc, 'fire')

    fire_only = fc.filter(ee.Filter.And(
        ee.Filter.eq('forest', 1),
        ee.Filter.eq('fire', 1)
    ))
    forest_only = fc.filter(ee.Filter.And(
        ee.Filter.eq('forest', 1),
        ee.Filter.eq('fire', 0)
    ))

    forest_sample = forest_only.randomColumn('rand', seed=RANDOM_SEED).filter(ee.Filter.lt('rand', FOREST_RANDOM_SAMPLE))
    fc_interest = fire_only.merge(forest_sample)

    # Conditional extraction to avoid processing empty collections
    result = ee.Algorithms.If(
        fc_interest.size().gt(0),
        extract_additional_data(fc_interest, tile_geom),
        ee.FeatureCollection([])  # Return empty collection if no features
    )

    return(result)






# def attach_polygon_properties_to_points(samples_fc, polygons_fc, property_names):
#     def add_props(point):
#         intersected = polygons_fc.filterBounds(point.geometry()).first()
#         def set_props(f):
#             return point.set({prop: f.get(prop) for prop in property_names})
#         return ee.Algorithms.If(intersected, set_props(intersected), point)
    
#     return samples_fc.map(add_props)

In [58]:

# used to create tile IDs for exports
def format_tile_id(f):
    # Use the feature's geometry directly instead of .bounds()
    geom4326 = f.geometry().transform('EPSG:4326', 1)
    coords = ee.List(geom4326.coordinates().get(0))
    
    # For a rectangle from coveringGrid, ring is usually [LL, LR, UR, UL, LL]
    upper_left = ee.List(coords.get(3))

    def format_coord(num):
        num = ee.Number(num)
        prefix = ee.Algorithms.If(num.gte(0), 'p', 'n')
        # encode 3 decimal places without a dot, as before
        abs_val = num.abs().multiply(1000).round().format('%d')
        safe_val = abs_val.replace('.', '')
        return ee.String(prefix).cat(safe_val)

    lon_str = format_coord(upper_left.get(0))
    lat_str = format_coord(upper_left.get(1))
    tile_id = lon_str.cat(lat_str)

    return f.set('tile_id', tile_id)

# # Create samples and export in batches
# # This needs to be done fully server-side to prevent creation
# # of the entire feature collection in memory, which hits memory limitations

aoi_bounds = aoi.geometry().bounds()
grid = aoi_bounds.coveringGrid(ee.Projection(TARGET_CRS).atScale(TILE_SIZE))
grid = grid.map(format_tile_id)


#####################################
# Loop through tiles and extract data
#####################################

grid_list = grid.toList(grid.size())
num_tiles = grid.size().getInfo()

# Operational export
for i in range(num_tiles):
#for i in range(3):
    tile = ee.Feature(grid_list.get(i))
    tile_geom = tile.geometry()
    tile_id = tile.get('tile_id').getInfo()

    fc_tile = extract(tile_geom)


    # Export tile points
    print(f"Submitted export for tile {tile_id}")
    task = ee.batch.Export.table.toDrive(
        collection=fc_tile,
        description=f'fc_tile_{AOI_READABLE}_{tile_id}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'fc_tile_{AOI_READABLE}_{tile_id}',
        fileFormat='CSV'
    )
    task.start()


# # TESTING 
# for i in range(5):
#     tile = ee.Feature(grid_list.get(i))
#     tile_geom = tile.geometry()
#     tile_id = tile.get('tile_id').getInfo()

#     fc_tile = extract(tile_geom)
#     # property_names = ee.FeatureCollection(fc_tile).first().propertyNames()
#     # print('Property names:', property_names.getInfo())
#     # print(ee.FeatureCollection(fc_tile).first().getInfo())

#     # Export tile points
#     print(f"Submitted export for tile {tile_id}")
#     task = ee.batch.Export.table.toDrive(
#         collection=fc_tile,
#         description=f'TEST2fc_tile_{AOI_READABLE}_{tile_id}',
#         folder=DRIVE_FOLDER,
#         fileNamePrefix=f'TEST2fc_tile_{AOI_READABLE}_{tile_id}',
#         fileFormat='CSV'
#     )
#     task.start()

#     # asset_id = f"{'projects/ee-tymc5571-multi-disturbance/assets'}/TEST2fc_tile_{AOI_READABLE}_{tile_id}"

#     # task = ee.batch.Export.table.toAsset(
#     #     collection=fc_tile,
#     #     description=f'TEST2fc_tile_{AOI_READABLE}_{tile_id}',
#     #     assetId=asset_id
#     # )
#     # task.start()


Submitted export for tile n19047p2556
Submitted export for tile n17853p2765
Submitted export for tile n16653p2964
Submitted export for tile n15446p3151
Submitted export for tile n21736p2967
Submitted export for tile n20544p3201
Submitted export for tile n19344p3424
Submitted export for tile n18137p3637
Submitted export for tile n16922p3838
Submitted export for tile n15701p4028
Submitted export for tile n22070p3830
Submitted export for tile n20864p4067
Submitted export for tile n19650p4294
Submitted export for tile n18428p4509
Submitted export for tile n17199p4713
Submitted export for tile n15962p4906
Submitted export for tile n21192p4934
Submitted export for tile n19964p5164
Submitted export for tile n18959p5342
Submitted export for tile n17716p5552
Submitted export for tile n16465p5750
Submitted export for tile n21761p5756
Submitted export for tile n20519p5992
Submitted export for tile n19269p6216
Submitted export for tile n18010p6429


In [59]:
print(grid.size().getInfo())

print(grid_list.size().getInfo())



25
25


In [60]:


Map = geemap.Map(center=[0, 0], zoom=2)
Map.addLayer(aoi, {}, 'aoi')
Map.addLayer(grid, {}, 'Grid Tiles')
Map.addLayer(
    usfs_mask_5070.clip(grid),
    {"min": 0, "max": 1, "palette": ["00000000", "00FF00"]},  # transparent → green
    "USFS Mask 5070"
)

Map.addLayer(
    combined_forest_mask_5070.clip(grid),
    {"min": 0, "max": 1, "palette": ["00000000", "FF8C00"]},  # transparent → dark orange
    "Forest Mask 5070"
)

Map.addLayer(
    fire_mask_5070.clip(grid),
    {"min": 0, "max": 1, "palette": ["00000000", "FF0000"]},  # transparent → red
    "Fire Mask 5070"
)

Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…